# Семинар 9. SVM.

In [ ]:
%matplotlib inline
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Определение SVM

#### Интуиция SVM

На рисунке представлена линейно-разделимая выборка. Слева изображены 2 прямые (гиперплоскости), разделяющие её. *Интуитивно*, синяя прямая лучше разделяет обучающую выборку, так как расстояние от объектов до неё больше, чем для красной и она лучше будет разделять другие примеры из того же распределения. Это мотивирует выбор разделяющей прямой с максимальным зазором; само по себе увеличение зазора не гарантирует лучший результат на любых новых данных. Теория подтверждает описанную интуицию [[1](https://www.academia.edu/download/86186744/lecture_3.pdf)].

*Метод опорных векторов (Support Vector Machine, SVM)* основан на идее построения оптимального разделения между классами.

In [ ]:
from sklearn.svm import SVC

# Both panels use the same separable sample.
intuition_X = np.array([
    [0.8, 3.0], [0.9, 2.0], [1.5, 1.8], [1.9, 0.6], [2.3, 1.1],
    [2.4, 5.1], [3.6, 4.2], [3.6, 3.2], [4.4, 3.7], [4.2, 2.5], [4.8, 1.3],
])
intuition_y = np.array([-1] * 5 + [1] * 6)
max_margin_model = SVC(kernel="linear", C=1e6, tol=1e-9).fit(intuition_X, intuition_y)
normal = max_margin_model.coef_[0]
intercept = max_margin_model.intercept_[0]

# A second separating line has a smaller geometric margin.
other_normal = np.array([1.0, 0.25])
negative_edge = (intuition_X[intuition_y == -1] @ other_normal).max()
positive_edge = (intuition_X[intuition_y == 1] @ other_normal).min()
other_intercept = -(negative_edge + 0.15 * (positive_edge - negative_edge))

fig, axes = plt.subplots(1, 2, figsize=(10, 5.4), sharex=True, sharey=True)
line_x = np.linspace(0.3, 5.5, 300)
for ax in axes:
    for label, color in [(-1, "royalblue"), (1, "orangered")]:
        points = intuition_X[intuition_y == label]
        ax.scatter(points[:, 0], points[:, 1], s=95, facecolors="white",
                   edgecolors=color, linewidths=1.8, zorder=4)
    ax.set(xlim=(0.3, 5.5), ylim=(0.2, 5.6), xlabel=r"$x_1$")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.2)

axes[0].set_ylabel(r"$x_2$")

svm_line = -(normal[0] * line_x + intercept) / normal[1]
axes[0].plot(line_x, svm_line, color="royalblue", linewidth=2)
axes[0].plot(line_x, -(other_normal[0] * line_x + other_intercept) / other_normal[1],
             color="orangered", linewidth=2)
axes[0].set_title("Разделяющие гиперплоскости")

axes[1].plot(line_x, svm_line, color="black", linewidth=2)
for level in [-1, 1]:
    axes[1].plot(line_x, (level - normal[0] * line_x - intercept) / normal[1],
                 color="black", linestyle="--", linewidth=1.5)
axes[1].fill_between(line_x, (-1 - normal[0] * line_x - intercept) / normal[1],
                     (1 - normal[0] * line_x - intercept) / normal[1], color="gold", alpha=0.15)
axes[1].set_title("Оптимальное разделение")
plt.tight_layout(pad=0.8, w_pad=0.4)
plt.subplots_adjust(wspace=0.04)
plt.show()


<Figure size 1000x540 with 2 Axes>

#### Постановка SVM

In [ ]:
# Orthogonal projection onto the hyperplane <w, x> = 0.
distance_w = np.array([1.0, 2.0])
point_a = np.array([2.5, 2.0])
projection_h = point_a - (distance_w @ point_a) * distance_w / (distance_w @ distance_w)
unit_normal = distance_w / np.linalg.norm(distance_w)
unit_tangent = np.array([distance_w[1], -distance_w[0]]) / np.linalg.norm(distance_w)

fig, ax = plt.subplots(figsize=(9, 6))
line_x = np.linspace(-2.8, 3.5, 200)
ax.plot(line_x, -distance_w[0] * line_x / distance_w[1], color="black", lw=2,
        label=r"$\langle w,x\rangle=0$")
ax.plot([projection_h[0], point_a[0]], [projection_h[1], point_a[1]], "k--", lw=2)
corner = np.array([projection_h + 0.18 * unit_tangent,
                   projection_h + 0.18 * (unit_tangent + unit_normal),
                   projection_h + 0.18 * unit_normal])
ax.plot(corner[:, 0], corner[:, 1], color="black", lw=1)
ax.scatter([0, point_a[0], projection_h[0]], [0, point_a[1], projection_h[1]], color="black", zorder=5)
ax.text(-0.25, -0.25, "$O$", fontsize=13)
ax.text(*(point_a + [0.12, 0.12]), "$x_A$", fontsize=14)
ax.text(*(projection_h + [0.1, -0.35]), "$H$", fontsize=13)
ax.set(xlim=(-2.8, 3.5), ylim=(-2, 3), xlabel=r"$x_1$", ylabel=r"$x_2$",
       title="Расстояние от точки до разделяющей гиперплоскости")
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.2)
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()


<Figure size 900x600 with 1 Axes>

Расстояние от точки $x_A$ до плоскости $\langle w,x \rangle = 0$:
$$
\rho(x^A, \langle w,x \rangle = 0) = \dfrac{|\langle w,x^A \rangle|}{\|w\|}
$$

Рассмотрим линейный классификатор $a(x) = sign(\langle w,x \rangle + b)$, $w \in \mathbb{R}^d, b \in \mathbb{R}$. Без ограничения общности положим $\min_{x\in X}|\langle w,x \rangle + b| = 1$. В противном случае соответствующим образом отнормируем параметры классификатора.

Расстояние от выборки X до гиперплоскости:
$$
\rho(X, a) = \min_{x \in X}\rho(x, a) = \dfrac{\min_{x\in X}|\langle w,x \rangle + b|}{\|w\|} = \dfrac{1}{\|w\|}
$$

Тогда, оптимизация строится вокруг максимизации зазора при условии корректного разделения:
$$
\begin{cases}
\dfrac{1}{2}\|w\|^2 \rightarrow \min_{w, b} \\
y_i(\langle w,x_i \rangle + b) \geqslant 1, & i=1,\ldots, l.
\end{cases}
$$
Выражение $M(x_i, y_i, w, b) = y_i(\langle w,x_i\rangle + b)$ называют **отступом классификации** (margin).

Если выборка линейно-неразделимая, то $\exists\, x_i \in X: y_i(\langle w,x_i \rangle + b) < 1$. Давайте *смягчим* ограничение, введя штраф $\xi_i$.
$$
\begin{cases}
\dfrac{1}{2}\|w\|^2 + C \sum_{i=1}^\ell \xi_i\rightarrow \min_{w, b, \xi} \\
y_i(\langle w,x_i \rangle + b) \geqslant 1 - \xi_i, & i=1,\ldots, \ell, \\
\xi_i \geqslant 0, & i=1,\ldots, \ell.
\end{cases} \tag{1}
$$

Полученная система является задачей *условной оптимизации*, её можно решить через условия *Каруша-Куна-Таккера*.

Решение задачи зависит от конкретных значений параметров системы. Их можно разбить на 3 категории, интерпретируемых через расстояние примера обучающей выборки до разделяющей гиперплоскости:


*   *Периферийный пример*
*   *Опорный граничный пример*
*   *Опорный нарушитель-пример*

In [ ]:
from matplotlib.lines import Line2D

rng = np.random.RandomState(42)
class_1 = rng.randn(6, 2) + [-2, 2]
class_2 = rng.randn(6, 2) + [2, -2]
x_vals = np.linspace(-3, 3, 100)
decision_boundary = 0.5 * x_vals

fig, ax = plt.subplots(figsize=(12, 8))
ax.set_facecolor("#f8f9fa")
ax.fill_between(x_vals, decision_boundary - 1, decision_boundary + 1, color="yellow", alpha=0.3)
ax.plot(x_vals, decision_boundary, "k-", linewidth=2.5)
for offset in [-1, 1]:
    ax.plot(x_vals, decision_boundary + offset, "k--", linewidth=2)

for points, boundary, violator, edge, boundary_fill, violation_fill in [
    (class_1, [-2.0, 0.0], [1.0, 0.0], "orangered", "orange", "red"),
    (class_2, [2.0, 0.0], [2.0, 0.5], "royalblue", "lightskyblue", "royalblue"),
]:
    ax.scatter(points[:, 0], points[:, 1], s=100, facecolors="none", edgecolors=edge, linewidths=1.5)
    ax.scatter(*boundary, s=130, facecolors=boundary_fill, edgecolors=edge, linewidths=2, zorder=3)
    ax.scatter(*violator, s=130, facecolors=violation_fill, edgecolors=edge, linewidths=2, zorder=3)

legend_handles = [
    Line2D([0], [0], marker="o", linestyle="none", markersize=9,
           markerfacecolor=fill, markeredgecolor="gray", label=label)
    for label, fill in [
        ("Периферийный", "none"),
        ("Опорный граничный", "lightgray"),
        ("Опорный нарушитель", "gray"),
    ]
]
ax.legend(handles=legend_handles, loc="upper right", fontsize=10)
ax.set(xlabel=r"$x_1$", ylabel=r"$x_2$", title="Классификация примеров в SVM")
ax.grid(linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()


<Figure size 1200x800 with 1 Axes>

#### Как полученная система уравнений связана с линейным классификатором?

Давайте вспомним задачу линейной классификации
$$
\dfrac{1}{N} \sum_{i=1}^N \mathbb{I}[M_i < 0] → \min_{w}
$$
Полученный функционал является кусочно-постоянным, его заменяют выпуклой функцией потерь. Рассмотрим hinge-loss $\max\{0, 1 - M_i\}$ с $\ell_2$-регуляризацией:
$$
\dfrac{1}{N} \sum_{i=1}^N \max\{0, 1 - y_i(\langle w, x_i\rangle + b)\} + \dfrac{\alpha}{2}\|w\|^2 → \min_{w, b} \tag{2}
$$
Ограничения типа неравенств в (1) даёт $\xi_i = \max\{0, 1 - y_i(\langle w, x_i\rangle + b)\}$. Подстваляя $\xi_i$ в первое уравнение системы (1), можно переписать:
$$
\dfrac{1}{2}\|w\|^2 + C \sum_{i=1}^\ell\max\{0, 1 - y_i(\langle w, x_i\rangle + b)\} \rightarrow \min_{w, b} \tag{3},
$$
что с точностью до домножения на $\frac{1}{CN}$ и переобозначения $\alpha = \frac{1}{CN}$ эквивалентно задаче (2).


In [ ]:
from sklearn.datasets import make_blobs
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y = make_blobs(n_samples=100, centers=2, random_state=42, cluster_std=1.0)
X = StandardScaler().fit_transform(X)
y = np.where(y == 0, -1, 1)
linear_models = {}
for C in [0.1, 10]:
    svm = SVC(kernel="linear", C=C, tol=1e-9)
    classifier = SGDClassifier(
        loss="hinge", penalty="l2", alpha=1 / (C * len(X)), random_state=42,
        max_iter=20000, tol=1e-7, n_iter_no_change=20,
    )
    svm.fit(X, y)
    classifier.fit(X, y)
    linear_models[C] = (svm, classifier)


In [ ]:
xx, yy = np.meshgrid(
    np.linspace(X[:, 0].min() - 0.4, X[:, 0].max() + 0.4, 200),
    np.linspace(X[:, 1].min() - 0.4, X[:, 1].max() + 0.4, 200),
)
grid = np.column_stack([xx.ravel(), yy.ravel()])


def plot_classifier(ax, model, title):
    for label, color in [(-1, "royalblue"), (1, "orangered")]:
        points = X[y == label]
        ax.scatter(points[:, 0], points[:, 1], s=20, color=color, label=f"Класс {label:+d}")
    scores = model.decision_function(grid).reshape(xx.shape)
    ax.contour(xx, yy, scores, levels=[-1, 0, 1], colors="black",
               linestyles=["--", "-", "--"], linewidths=[1, 1.5, 1])
    ax.set(title=title, xlim=(xx.min(), xx.max()), ylim=(yy.min(), yy.max()),
           xlabel=r"$x_1$", ylabel=r"$x_2$")
    ax.grid(alpha=0.2)


fig, axes = plt.subplots(2, 2, figsize=(9, 6), sharex=True, sharey=True)
for column, (C, models) in enumerate(linear_models.items()):
    svm, classifier = models
    plot_classifier(axes[0, column], svm, f"SVM, C = {C:g}")
    plot_classifier(axes[1, column], classifier, f"Линейный классификатор (SGD), C = {C:g}")
    axes[0, column].scatter(svm.support_vectors_[:, 0], svm.support_vectors_[:, 1],
                         s=65, facecolors="none", edgecolors="black", label="Опорные векторы")
    for ax in axes[:, column]:
        ax.legend(loc="upper left", fontsize=7)
        ax.title.set_fontsize(10)

plt.tight_layout()
plt.show()


<Figure size 900x600 with 4 Axes>

### SVM unbalanced case

Давайте аналогично взвешиванию функции потерь взвешивать вклады от $\xi_i$ для разных $y_i$:
$$
\dfrac{1}{2}\|w\|^2 + C \sum_{i=1}^{\ell}\xi_i \rightarrow \min_{w,b,\xi}
\quad\Rightarrow\quad
\dfrac{1}{2}\|w\|^2 + C_1 \sum_{y_i=1}\xi_i + C_2 \sum_{y_i=-1}\xi_i \rightarrow \min_{w,b,\xi}.
$$

In [ ]:
# Authors: The scikit-learn developers
# SPDX-License-Identifier: BSD-3-Clause

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.metrics import recall_score
from sklearn.svm import SVC

n_samples_1, n_samples_2 = 1000, 100
X, y = make_blobs(
    n_samples=[n_samples_1, n_samples_2],
    centers=[[0.0, 0.0], [2.0, 2.0]],
    cluster_std=[1.5, 0.5],
    random_state=0,
    shuffle=False,
)
clf = SVC(kernel="linear", C=1.0).fit(X, y)
wclf = SVC(kernel="linear", C=1.0, class_weight={1: n_samples_1 / n_samples_2}).fit(X, y)

g_mean = np.sqrt(np.prod(recall_score(y, clf.predict(X), average=None)))
weighted_g_mean = np.sqrt(np.prod(recall_score(y, wclf.predict(X), average=None)))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
x_limits = (X[:, 0].min() - 0.5, X[:, 0].max() + 0.5)
y_limits = (X[:, 1].min() - 0.5, X[:, 1].max() + 0.5)
line_x = np.linspace(*x_limits, 200)
boundaries = []
for model, name, color, style, score in [
    (clf, "non weighted", "black", "-", g_mean),
    (wclf, "weighted", "crimson", "--", weighted_g_mean),
]:
    w = model.coef_[0]
    boundary = -(w[0] * line_x + model.intercept_[0]) / w[1]
    boundaries.append(boundary)
    ax.plot(line_x, boundary, color=color, linestyle=style, linewidth=2.5,
            label=f"{name}, G-mean = {score:.3f}", zorder=3)

ax.fill_between(line_x, *boundaries, color="gray", alpha=0.15, zorder=0)
ax.scatter(X[:, 0], X[:, 1], c=np.where(y == 0, "royalblue", "orangered"),
           s=16, alpha=0.5, edgecolors="none", zorder=2)
ax.set(xlim=x_limits, ylim=y_limits, xlabel=r"$x_1$", ylabel=r"$x_2$")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()


<Figure size 800x600 with 1 Axes>

### Kernel SVM

#### Переход в спрямляющее пространство

Если классы нельзя разделить гиперплоскостью в исходных признаках, применим SVM к новым признакам $\psi(x)$. Граница
$$
\langle w,\psi(x)\rangle+b=0
$$
линейна по $\psi(x)$, но может быть нелинейной по $x$.


Для этого необходимо, чтобы функция $\mathcal{K}(x_i, x_j)$ была **ядром** (англ. kernel):
$$
\mathcal{K}: \mathcal{X}\times\mathcal{X} \rightarrow \mathbb{R}, \quad \mathcal{K}(x_i, x_j) = \langle \psi(x_i), \psi(x_j) \rangle,
$$
при некотором $\psi: \mathcal{X} \rightarrow H$, где $H$ -- пространство со скалярным произведением.

Примеры распространённых ядерных функций:
* Константа $ \mathcal{K}_{C}(x,y) = C$, $C\geqslant 0$;
* Линейное $\mathcal{K}_{L}(x,y) =  x^\top y$;
* Полиномиальное $\mathcal{K}_{d, c}(x,y) = (x^\top y + c)^d$, $c\geqslant 0$, $d\in\mathbb{N}$;
* Радиально-Базисная Функция (РБФ) $\mathcal{K}_\gamma(x, y) = \exp\left\{-\gamma\|x-y\|^2\right\}$, $\gamma>0$;

Рассмотрим концентрические окружности для демонстрации нелинейности.

In [ ]:
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.svm import SVC

rng = np.random.default_rng(42)
circle_y = np.repeat([0, 1], 100)
angles = rng.uniform(0, 2 * np.pi, len(circle_y))
radii = np.where(circle_y == 0, 1.0, 2.0) + rng.uniform(-0.08, 0.08, len(circle_y))
circle_X = radii[:, None] * np.column_stack([np.cos(angles), np.sin(angles)])

kernel_parameters = {
    "Linear": {"kernel": "linear"},
    "Polynomial": {"kernel": "poly", "degree": 2, "gamma": 1, "coef0": 0},
    "RBF": {"kernel": "rbf", "gamma": 1},
}
circle_models = {
    name: SVC(C=10, **parameters).fit(circle_X, circle_y)
    for name, parameters in kernel_parameters.items()
}


In [ ]:
grid_x, grid_y = np.meshgrid(np.linspace(-2.4, 2.4, 160), np.linspace(-2.4, 2.4, 160))
grid = np.column_stack([grid_x.ravel(), grid_y.ravel()])

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
for ax, (name, model) in zip(axes, circle_models.items()):
    scores = model.decision_function(grid).reshape(grid_x.shape)
    ax.contourf(
        grid_x,
        grid_y,
        scores > 0,
        levels=[-0.5, 0.5, 1.5],
        colors=["royalblue", "darkorange"],
        alpha=0.12,
    )
    ax.contour(grid_x, grid_y, scores, levels=[0], colors="black", linewidths=1.2)
    for label, color in [(0, "royalblue"), (1, "darkorange")]:
        ax.scatter(*circle_X[circle_y == label].T, color=color, s=12, alpha=0.8)
    ax.set(title=name, xlabel=r"$x_1$", ylabel=r"$x_2$", aspect="equal")
fig.tight_layout()
plt.show()


<Figure size 1100x350 with 3 Axes>

Перейдём в спрямляющее пространства.

Для квадратичного ядра $\mathcal{K}_{2, 0}(x,y) = (x^\top y)^2$ используем отображение из лекции:
$$
\psi(x_1,x_2)=\left(x_1^2,x_2^2,\sqrt{2}x_1x_2\right),\qquad
\langle\psi(x),\psi(x')\rangle=\langle x,x'\rangle^2.
$$
Концентрические окружности разделяются плоскостью в $\mathbb R^3$. В исходном пространстве этой плоскости соответствует квадратичная граница.

In [ ]:
circle_features = np.column_stack([
    circle_X[:, 0] ** 2,
    circle_X[:, 1] ** 2,
    np.sqrt(2) * circle_X[:, 0] * circle_X[:, 1],
])
plane_model = SVC(kernel="linear", C=10).fit(circle_features, circle_y)


In [ ]:
fig = plt.figure(figsize=(10, 4.3))
ax_original = fig.add_subplot(121)
ax_lifted = fig.add_subplot(122, projection="3d")
for label, color in [(0, "royalblue"), (1, "darkorange")]:
    mask = circle_y == label
    ax_original.scatter(*circle_X[mask].T, color=color, s=18, alpha=0.8)
    ax_lifted.scatter(*circle_features[mask].T, color=color, s=15, alpha=0.8, depthshade=False)

plane_x, plane_z = np.meshgrid(np.linspace(0, 2.7, 30), np.linspace(-2.7, 2.7, 30))
w = plane_model.coef_[0]
plane_y = -(plane_model.intercept_[0] + w[0] * plane_x + w[2] * plane_z) / w[1]
plane_y[(plane_y < 0) | (plane_y > 2.7)] = np.nan
ax_lifted.plot_surface(plane_x, plane_y, plane_z, color="gray", alpha=0.26, linewidth=0, shade=False)
ax_original.set(xlabel=r"$x_1$", ylabel=r"$x_2$", title="Исходное пространство", aspect="equal")
ax_lifted.set(xlabel=r"$x_1^2$", ylabel=r"$x_2^2$", title="Спрямляющее пространство")
ax_lifted.view_init(elev=24, azim=-60)
ax_lifted.text2D(1.10, 0.5, r"$\sqrt{2}x_1x_2$", transform=ax_lifted.transAxes, rotation=90, va="center", clip_on=False)
fig.subplots_adjust(left=0.07, right=0.91, bottom=0.12, top=0.9, wspace=0.2)
plt.show()


<Figure size 1000x430 with 2 Axes>

#### Двойственная задача

Задача оптимизации с ограничениями
$$
\begin{cases}
\min_{x\in Q} f(x) \\
h_i(x)\leqslant 0, & i=1,\ldots,p
\end{cases} \Rightarrow \quad
\mathcal L(x,\mu)=f(x)+\sum_{i=1}^{p}\mu_i h_i(x) \rightarrow \min_{x\in Q},
$$
$Q$ — область определения переменных, $\mu_i\geqslant0$ — множители Лагранжа.

Двойственность по Лагранжу
$$
\max_{\mu\geqslant 0}G(\mu),\qquad G(\mu)=\inf_{x\in Q}\mathcal L(x,\mu).
$$

При фиксированных множителях минимизируем лагранжиан по $x$. Двойственная задача выбирает наихудшие для этой минимизации веса ограничений: при них минимум лагранжиана наибольший.


В методе опорных векторов ограничениями оптимизационной задачи являются неравенства:
$$
y_i(\langle w,x_i\rangle+b)\geqslant1-\xi_i,\qquad \xi_i\geqslant0.
$$
Для *периферийных* примеров $y_i(\langle w,x_i\rangle+b)>1$, поэтому ограничение выполнено при любом $\xi_i\geqslant0$.

Минимизируя лагранжиан по $w$, $b$ и $\xi$, получаем двойственную задачу. Данные входят в неё только через скалярные произведения; заменяя их ядром, получаем постановку для Kernel SVM:
$$
\begin{cases}
\displaystyle\sum_i\mu_i-\frac12\sum_{i,j}\mu_i\mu_j y_i y_j\langle x_i,x_j\rangle\to\max_{\mu},\\
0\leqslant\mu_i\leqslant C,\\
\sum_i\mu_i y_i=0
\end{cases}
\quad\Rightarrow\quad
\begin{cases}
\displaystyle\sum_i\mu_i-\frac12\sum_{i,j}\mu_i\mu_j y_i y_j\mathcal K(x_i,x_j)\to\max_{\mu},\\
0\leqslant\mu_i\leqslant C,\\
\sum_i\mu_i y_i=0.
\end{cases}\tag{1}
$$
Интуиция подтверждается решением: $\mu_i=0$ для периферийных примеров $y_i (\langle w,x_i\rangle+b)>1$.

SVM является линейным классификатором (оптимизируемым hinge-loss с $\ell_2$-регуляризацией). Решению задачи соответствует решающая функция
$$
f(x)= \langle w,\psi(x)\rangle + b \overset{w = \sum_i \mu_i y_i \psi(x_i)}{=} \sum_{i:\,\mu_i>0}\mu_i y_i\mathcal K(x_i,x)+b.
$$

Рассмотрим пример с концентрическими окружностями и RFB ядром $\mathcal{K}_\gamma(x, y) = \exp\left\{-\|x-y\|^2\right\}$, $\gamma = 1$ и C=10.

Для произвольной точки $x=(1.5,0)$ предсказание определяется опорными векторами. Вклад объекта $x_i$ в предсказание для $x$ равен
$$
c_i(x)=\mu_i y_i\mathcal K(x_i,x)
=\mu_i y_i\exp\!\left(-\|x_i-x\|^2\right),
$$
где $y_i$ — класс объекта $x_i$.



In [ ]:
contribution_model = SVC(kernel="rbf", C=10, gamma=1, tol=1e-8).fit(circle_X, circle_y)
support_ids = contribution_model.support_
query = np.array([[1.5, 0]])
contributions = (
    contribution_model.dual_coef_[0]
    * rbf_kernel(contribution_model.support_vectors_, query, gamma=1).ravel()
)
original_score = contribution_model.decision_function(query)[0]


In [ ]:
scores = contribution_model.decision_function(grid).reshape(grid_x.shape)
largest_terms = np.argsort(np.abs(contributions))[-6:]
selected_ids = contribution_model.support_[largest_terms]
bar_values = np.r_[
    contributions[largest_terms],
    contributions.sum() - contributions[largest_terms].sum(),
    contribution_model.intercept_[0],
]
bar_labels = [str(index + 1) for index in selected_ids] + ["Остальные", "b"]
bar_colors = np.where(bar_values > 0, "darkorange", "royalblue").tolist()
bar_colors[-1] = "gray"

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].contour(grid_x, grid_y, scores, levels=[0], colors="black", linewidths=1)
for label, color in [(0, "royalblue"), (1, "darkorange")]:
    mask = circle_y[support_ids] == label
    axes[0].scatter(*contribution_model.support_vectors_[mask].T, color=color, s=25)
axes[0].scatter(*circle_X[selected_ids].T, facecolors="none", edgecolors="gray", s=80)
axes[0].scatter(*query.T, color="black", marker="*", s=100)
for index in selected_ids:
    axes[0].annotate(
        str(index + 1), circle_X[index], xytext=(4, 4), textcoords="offset points", fontsize=8
    )
axes[0].set(xlabel=r"$x_1$", ylabel=r"$x_2$", title=r"$x=(1.5,0)$", aspect="equal")
axes[1].barh(bar_labels, bar_values, color=bar_colors)
axes[1].axvline(0, color="gray", linewidth=0.8)
axes[1].set(xlabel="Вклад в решающую функцию", title=rf"$f(x)={original_score:.3f}$")
fig.tight_layout()
plt.show()


<Figure size 900x350 with 2 Axes>

#### Операции с функциями ядра

<details>
<summary>Операции с функциями ядра</summary>

По теореме Мерсера положительная полуопределённость матриц Грама позволяет проверить, что функция задаёт ядро. Для суммы и поточечного произведения ядер это свойство сохраняется: если $\mathcal K_1$ и $\mathcal K_2$ — ядра, то при $a,b\geqslant0$
$$
\mathcal K_{\mathrm{sum}}(x,x')=a\mathcal K_1(x,x')+b\mathcal K_2(x,x'),
\qquad
\mathcal K_{\mathrm{prod}}(x,x')=\mathcal K_1(x,x')\mathcal K_2(x,x')
$$
также являются ядрами.

</details>


На графиках ниже один аргумент ядра фиксирован: $t'=1$. По горизонтали меняется $t$, по вертикали отложено значение $\mathcal K(t,1)$.

In [ ]:
t = np.linspace(-3, 5, 200)
reference = 1
kernel_values = [
    ("Linear", t * reference),
    ("Periodic", np.exp(-2 * np.sin(np.pi * (t - reference) / 2) ** 2)),
    ("RBF", np.exp(-0.5 * (t - reference) ** 2)),
]

fig, axes = plt.subplots(2, 3, figsize=(9, 5))
for column, ((name_1, values_1), (name_2, values_2)) in enumerate(
    itertools.combinations(kernel_values, 2)
):
    for row, symbol, values in [(0, "+", values_1 + values_2), (1, "*", values_1 * values_2)]:
        axes[row, column].plot(t, values)
        axes[row, column].set(
            title=f"{name_1} {symbol} {name_2}", xlabel=r"$t$", ylabel=r"$\mathcal{K}(t,1)$"
        )
fig.tight_layout()
plt.show()


<Figure size 900x500 with 6 Axes>

### Support Vector Regression

В $\varepsilon$-SVR цель $y_i\in\mathbb R$, предсказание $f(x)=\langle w,\psi(x)\rangle+b$. Ошибка в пределах $\varepsilon$ не штрафуется:
$$
L_\varepsilon(y,f)=\max\{0,\ |y-f|-\varepsilon\},\qquad
\frac12\|w\|^2+C\sum_{i=1}^{\ell}L_\varepsilon(y_i,f(x_i))\longrightarrow\min_{w,b}.
$$
Эквивалентная задача с переменными отклонений:
$$
\begin{cases}
\dfrac12\|w\|^2+C\displaystyle\sum_{i=1}^{\ell}(\xi_i+\xi_i^*)\longrightarrow\min_{w,b,\xi,\xi^*},\\[2pt]
y_i-f(x_i)\leqslant\varepsilon+\xi_i,\\
f(x_i)-y_i\leqslant\varepsilon+\xi_i^*,\\
\xi_i,\xi_i^*\geqslant0,\quad i=1,\ldots,\ell.
\end{cases}
$$
Здесь $C>0$, $\varepsilon\geqslant0$; $\xi_i$ и $\xi_i^*$ — превышения допустимой ошибки снизу и сверху. В SVC штрафуется нарушение условия $y_i f(x_i)\geqslant1$; в SVR — выход за двухстороннюю полосу $|y_i-f(x_i)|\leqslant\varepsilon$. $\varepsilon$ измеряется в единицах цели; полная ширина полосы равна $2\varepsilon$. Объекты строго внутри неё имеют нулевые двойственные коэффициенты.


Обучим линейную SVR и выделим три типа объектов: внутри $\varepsilon$-полосы, на её границе и за её пределами. Красные отрезки показывают штрафуемое превышение $\max\{0,\ |y_i-f(x_i)|-\varepsilon\}$; внутри полосы оно равно нулю.

In [ ]:
from sklearn.svm import SVR

svr_X = np.arange(12.0)[:, None]
svr_y = 0.5 * svr_X[:, 0] + 1 + np.array([
    0, 0.1, -0.8, 0.15, 0.8, 0, -0.1, 0, -0.9, 0.15, 0.7, 0,
])
svr_model = SVR(kernel="linear", C=1, epsilon=0.3, tol=1e-9).fit(svr_X, svr_y)
svr_prediction = svr_model.predict(svr_X)
svr_residual = svr_y - svr_prediction
svr_error = np.abs(svr_residual)
svr_epsilon = svr_model.epsilon


fig, ax = plt.subplots(figsize=(9, 4.2))
ax.set_facecolor("#f8f9fa")
ax.fill_between(
    svr_X[:, 0], svr_prediction - svr_epsilon, svr_prediction + svr_epsilon,
    color="gold", alpha=0.18,
)
ax.plot(svr_X[:, 0], svr_prediction, color="black", linewidth=1.8)
for offset, label in [
    (-svr_epsilon, r"$f(x)-\varepsilon$"),
    (0, r"$f(x)$"),
    (svr_epsilon, r"$f(x)+\varepsilon$"),
]:
    if offset:
        ax.plot(svr_X[:, 0], svr_prediction + offset, "k--", linewidth=1)
    ax.text(11.25, svr_prediction[-1] + offset, label, va="center", fontsize=10)

for mask, fill, edge, label in [
    (svr_error < svr_epsilon - 1e-6, "white", "gray", "Внутри полосы"),
    (np.isclose(svr_error, svr_epsilon, atol=1e-6), "orange", "darkorange", "Опорный граничный"),
    (svr_error > svr_epsilon + 1e-6, "tomato", "firebrick", "Опорный нарушитель"),
]:
    ax.scatter(svr_X[mask, 0], svr_y[mask], s=65, facecolors=fill,
               edgecolors=edge, linewidths=1.5, label=label, zorder=3)

outside = svr_error > svr_epsilon + 1e-6
tube_edge = svr_prediction + np.sign(svr_residual) * svr_epsilon
ax.vlines(svr_X[outside, 0], tube_edge[outside], svr_y[outside], color="firebrick")
violation = np.argmin(svr_residual)
ax.annotate(
    r"$\xi_i^*$",
    (svr_X[violation, 0], (tube_edge[violation] + svr_y[violation]) / 2),
    xytext=(10, 0), textcoords="offset points", va="center", color="firebrick",
)
ax.set(xlabel=r"$x$", ylabel=r"$y$", xlim=(-0.5, 13), title="Примеры в ε-SVR")
ax.spines[["top", "right"]].set_visible(False)
ax.set_axisbelow(True)
ax.grid(alpha=0.15)
ax.legend(loc="upper left", fontsize=9, frameon=False)
fig.tight_layout()
plt.show()


<Figure size 900x420 with 1 Axes>

### Examples

#### AirPassengers

[AirPassengers](https://stat.ethz.ch/R-manual/R-devel/library/datasets/html/AirPassengers.html): ежемесячное число международных авиапассажиров в 1949–1960 годах, в тысячах человек. Обучение: 1949–1956 годы; валидация: 1957–1958; тест: 1959–1960. Параметры выбираем по MAE на валидации, затем переоцениваем признаки и SVR по 1949–1958 годам.

В CSV столбец `time` содержит календарный год с дробной частью: $1949$, $1949+1/12$, $1949+2/12$, $\ldots$. Вычитаем $1949$ и получаем признак $t$ — число лет с января 1949 года: $0$, $1/12\approx0.0833$, $2/12$, $\ldots$. Один шаг наблюдений — месяц. Цель $y$ — число пассажиров в тысячах.

In [ ]:
from functools import partial

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

airline = pd.read_csv(
    "https://vincentarelbundock.github.io/Rdatasets/csv/datasets/AirPassengers.csv"
)
airline_X = (airline["time"].to_numpy() - 1949)[:, None]
airline_y = airline["value"].to_numpy()
fit_X, fit_y = airline_X[:96], airline_y[:96]
validation_X, validation_y = airline_X[96:120], airline_y[96:120]
train_X, train_y = airline_X[:120], airline_y[:120]
test_X, test_y = airline_X[120:], airline_y[120:]
forecast_X = np.linspace(0, airline_X[-1, 0], 600)[:, None]

In [ ]:
pd.DataFrame({"time": airline["time"], "t": airline_X[:, 0], "y": airline_y})

,time,t,y
0,1949.000000,0.000000,112
1,1949.083333,0.083333,118
2,1949.166667,0.166667,132
3,1949.250000,0.250000,129
4,1949.333333,0.333333,121
...,...,...,...
139,1960.583333,11.583333,606
140,1960.666667,11.666667,508
141,1960.750000,11.750000,461
142,1960.833333,11.833333,390


In [ ]:
plt.figure(figsize=(9, 3))
plt.plot(airline["time"], airline_y, color="black", linewidth=1)
plt.axvline(1957, color="gray", linestyle=":", label="Начало валидации")
plt.axvline(1959, color="gray", linestyle="--", label="Начало теста")
plt.xlabel("Год")
plt.ylabel("Тысячи пассажиров")
plt.legend()
plt.tight_layout()
plt.show()

<Figure size 900x300 with 1 Axes>

Стандартизуем цель: $\widetilde y=(y-\overline y)/s_y$. Для подбора параметров среднее и стандартное отклонение вычисляем по 1949–1956 годам; после подбора — по 1949–1958 годам для итогового обучения. Признак $t=\mathrm{time}-1949$ не стандартизуем; его единица — год, шаг наблюдений — $1/12$ года. Предсказания и ошибки переводим обратно в тысячи пассажиров. Прогноз в исходных единицах: $\hat y(t)=\overline y+s_y f(t)$.

In [ ]:
fit_scaler = StandardScaler().fit(fit_y[:, None])
fit_target = fit_scaler.transform(fit_y[:, None]).ravel()
validation_target = fit_scaler.transform(validation_y[:, None]).ravel()

target_scaler = StandardScaler().fit(train_y[:, None])
train_target = target_scaler.transform(train_y[:, None]).ravel()

Переберём $C\in\{0.1,1,10,100\}$ и $\varepsilon\in\{0.01,0.1,0.3,1\}$. $\varepsilon$ — полуширина полосы в единицах стандартного отклонения цели, $C$ — вес штрафа за выход из неё. В исходных единицах полоса имеет вид $\hat y(t)\pm\varepsilon s_y$. Выбираем параметры по MAE на валидации; при смене ядра повторяем подбор вместе с его параметрами.

In [ ]:
svr_grid = {"C": [0.1, 1, 10, 100], "epsilon": [0.01, 0.1, 0.3, 1]}

In [ ]:
def evaluate_airline(model, name):
    prediction = target_scaler.inverse_transform(model.predict(test_X)[:, None]).ravel()
    forecast = target_scaler.inverse_transform(model.predict(forecast_X)[:, None]).ravel()
    mae = mean_absolute_error(test_y, prediction)
    rmse = np.sqrt(mean_squared_error(test_y, prediction))
    epsilon = model.epsilon * target_scaler.scale_[0]
    print(
        f"{name}: C={model.C:g}, ε={model.epsilon:g}; "
        f"полуширина полосы {epsilon:.1f} тыс."
    )
    return forecast, mae, rmse, epsilon

In [ ]:
def plot_airline(result, name, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 3))
    forecast, mae, rmse, epsilon = result
    years = forecast_X[:, 0] + 1949
    ax.fill_between(
        years, forecast - epsilon, forecast + epsilon,
        color="royalblue", alpha=0.12, label=r"Полоса $\pm\varepsilon$",
    )
    for edge in [forecast - epsilon, forecast + epsilon]:
        ax.plot(years, edge, color="royalblue", linestyle="--", linewidth=0.8, alpha=0.6)
    ax.scatter(
        airline["time"][:120], train_y,
        color="gray", s=12, alpha=0.65, label="Обучение",
    )
    ax.scatter(airline["time"][120:], test_y, color="darkorange", s=16, label="Тест")
    ax.plot(years, forecast, color="royalblue", label="SVR")
    ax.axvline(1959, color="black", linestyle="--", linewidth=1)
    ax.set(
        title=f"{name}: MAE={mae:.1f}, RMSE={rmse:.1f}",
        xlabel="Год", ylabel="Тысячи пассажиров", ylim=(50, 700),
    )
    ax.legend(fontsize=8)
    return ax

**1. RBF.** Пусть $t$ — число лет с января 1949 года. Для начала возьмём радиально-базисное ядро
$$
\mathcal K_{\mathrm{rbf}}(t,s)=\exp\left[-\frac{(t-s)^2}{2\ell^2}\right],
\qquad \gamma=\frac{1}{2\ell^2}.
$$
$\ell>0$ — масштаб расстояния в годах: при $|t-s|=\ell$ сходство равно $e^{-1/2}$. Чем больше $\ell$, тем медленнее оно убывает.

Проверим $\ell\in\{0.5,1,2,4,8\}$; в `SVR` передадим соответствующее $\gamma$. Вместе с $C$ и $\varepsilon$ выберем его по MAE на валидации.

In [ ]:
rbf_candidates = []
rbf_grid = {
    **svr_grid,
    "gamma": [1 / (2 * length**2) for length in [0.5, 1, 2, 4, 8]],
}
for parameters in ParameterGrid(rbf_grid):
    model = SVR(kernel="rbf", **parameters).fit(fit_X, fit_target)
    error = mean_absolute_error(validation_target, model.predict(validation_X))
    rbf_candidates.append((error, parameters))
_, rbf_parameters = min(rbf_candidates, key=lambda item: item[0])

rbf_forecaster = SVR(kernel="rbf", **rbf_parameters).fit(train_X, train_target)
rbf_length = np.sqrt(1 / (2 * rbf_parameters["gamma"]))
print(f"ℓ = {rbf_length:g} лет; γ = {rbf_parameters['gamma']:g}")
rbf_result = evaluate_airline(rbf_forecaster, "RBF")

ℓ = 4 лет; γ = 0.03125
RBF: C=100, ε=1; полуширина полосы 94.5 тыс.


In [ ]:
plot_airline(rbf_result, "RBF")
plt.tight_layout()
plt.show()


<Figure size 900x300 with 1 Axes>

Пример работы functools.partial

In [ ]:
def multiply(x, factor):
    return factor * x


double = partial(multiply, factor=2)
double(3)

6

**2. Линейный тренд.** На зависимости присутствует явный возрастающий тренд, поэтому логично воспользоваться $\mathcal{K}_{\mathrm{lin}}(x,y) =  x^\top y$.

Оценим скорость роста методом наименьших квадратов:
$$
(\hat a,\hat v)=\arg\min_{a,v}\sum_{i\in\mathrm{train}}(y_i-a-vt_i)^2.
$$
$$
\Rightarrow\quad
\hat v=\frac{\sum_i(t_i-\bar t)(y_i-\bar y)}{\sum_i(t_i-\bar t)^2},\qquad
\hat a=\bar y-\hat v\bar t.
$$
В единицах стандартизованной цели:
$$
\frac{\hat a+\hat vt-\bar y}{s_y}
=\frac{\hat a-\bar y}{s_y}+\psi_{\mathrm{lin}}(t),\qquad
\psi_{\mathrm{lin}}(t)=\frac{\hat vt}{s_y}.
$$
Линейное ядро в этом пространстве:
$$
\mathcal K_{\mathrm{lin}}^{(t)}(t,s)
=\langle\psi_{\mathrm{lin}}(t),\psi_{\mathrm{lin}}(s)\rangle
=\frac{\hat v t}{s_y}\frac{\hat v s}{s_y}=\frac{\hat v^2}{s_y^2}ts.
$$
Прогноз SVR:
$$
f(t)=w_{\mathrm{lin}}\psi_{\mathrm{lin}}(t)+b,\qquad
\widehat y(t)=\bar y+s_yf(t)=\bar y+s_yb+w_{\mathrm{lin}}\hat vt.
$$
Коэффициент $w_{\mathrm{lin}}$ и свободный член $b$ оценивает SVR; МНК задаёт масштаб признака.

`fit_linear` использует оценки за 1949–1956 годы для подбора параметров. `airline_linear` — оценки за 1949–1958 годы для итогового обучения; в каждом случае используется соответствующий $s_y$.

In [ ]:
def linear_kernel(X, Y, scale):
    return scale**2 * (X @ Y.T)


fit_trend_slope, fit_trend_offset = np.polyfit(fit_X[:, 0], fit_y, 1)
trend_slope, trend_offset = np.polyfit(train_X[:, 0], train_y, 1)
fit_linear = partial(linear_kernel, scale=fit_trend_slope / fit_scaler.scale_[0])
airline_linear = partial(linear_kernel, scale=trend_slope / target_scaler.scale_[0])
linear_candidates = []
for parameters in ParameterGrid(svr_grid):
    model = SVR(kernel=fit_linear, **parameters).fit(fit_X, fit_target)
    error = mean_absolute_error(validation_target, model.predict(validation_X))
    linear_candidates.append((error, parameters))
_, linear_parameters = min(linear_candidates, key=lambda item: item[0])

linear_forecaster = SVR(kernel=airline_linear, **linear_parameters).fit(train_X, train_target)
print(f"Скорость тренда по МНК: {trend_slope:.2f} тыс./год")
linear_result = evaluate_airline(linear_forecaster, "Linear")


Скорость тренда по МНК: 29.94 тыс./год
Linear: C=100, ε=0.3; полуширина полосы 28.4 тыс.


In [ ]:
plot_airline(linear_result, "Linear")
plt.tight_layout()
plt.show()


<Figure size 900x300 with 1 Axes>

**3. Периодичность.** Колебания повторяются по сезонам календарного года. Зададим период по этому априорному знанию: $P=12$ месяцев, то есть $P=1$ в единицах признака $t$.
$$
\mathcal K_{\mathrm{per}}(t,s)
=\exp\left[-\frac{2\sin^2(\pi(t-s)/P)}{\ell_{\mathrm{per}}^2}\right]
=\langle\psi_{\mathrm{per}}(t),\psi_{\mathrm{per}}(s)\rangle.
$$
$\ell_{\mathrm{per}}$ задаёт сходство разных фаз сезона: малое значение допускает более резкие изменения, большое — более плавный профиль. Выберем $C$, $\varepsilon$ и $\ell_{\mathrm{per}}$

Центрируем признаки по 12 месячным фазам года:
$$
s_m=\frac{mP}{12},\quad m=0,\ldots,11.
$$
$$
\bar\psi_{\mathrm{per}}=\frac1{12}\sum_{m=0}^{11}\psi_{\mathrm{per}}(s_m),\qquad
\widetilde\psi_{\mathrm{per}}(t)=\psi_{\mathrm{per}}(t)-\bar\psi_{\mathrm{per}}.
$$
$$
g(t)=\langle w_{\mathrm{per}},\widetilde\psi_{\mathrm{per}}(t)\rangle,\qquad
f(t)=b+g(t).
$$
$$
\Rightarrow\quad
\frac1{12}\sum_{m=0}^{11}g(s_m)=0,\qquad
\frac1{12}\sum_{m=0}^{11}f(s_m)=b.
$$
Соответствующее центрированное ядро:
$$
\begin{aligned}
\widetilde{\mathcal K}_{\mathrm{per}}(t,s)
&=\langle\widetilde\psi_{\mathrm{per}}(t),\widetilde\psi_{\mathrm{per}}(s)\rangle\\
&=\mathcal K_{\mathrm{per}}(t,s)
-\frac1{12}\sum_{m=0}^{11}\mathcal K_{\mathrm{per}}(t,s_m)\\
&\quad-\frac1{12}\sum_{m=0}^{11}\mathcal K_{\mathrm{per}}(s_m,s)
+\frac1{144}\sum_{m,n=0}^{11}\mathcal K_{\mathrm{per}}(s_m,s_n).
\end{aligned}
$$
Это ядро повторяет сезонный профиль, но не продолжает рост среднего уровня.

In [ ]:
def periodic_kernel(X, Y, period, length_scale):
    def similarity(A, B):
        phase = np.pi * (A - B.T) / period
        return np.exp(-2 * np.sin(phase) ** 2 / length_scale**2)

    reference = np.arange(12)[:, None] * period / 12
    return (
        similarity(X, Y)
        - similarity(X, reference).mean(axis=1)[:, None]
        - similarity(reference, Y).mean(axis=0)[None, :]
        + similarity(reference, reference).mean()
    )


season_period = 1.0

In [ ]:
periodic_candidates = []
periodic_grid = {**svr_grid, "length_scale": [0.25, 0.5, 1, 2]}
for parameters in ParameterGrid(periodic_grid):
    kernel = partial(
        periodic_kernel,
        period=season_period,
        length_scale=parameters["length_scale"],
    )
    model = SVR(kernel=kernel, C=parameters["C"], epsilon=parameters["epsilon"]).fit(
        fit_X, fit_target
    )
    error = mean_absolute_error(validation_target, model.predict(validation_X))
    periodic_candidates.append((error, parameters))
_, periodic_parameters = min(periodic_candidates, key=lambda item: item[0])

airline_periodic = partial(
    periodic_kernel,
    period=season_period,
    length_scale=periodic_parameters["length_scale"],
)
periodic_forecaster = SVR(
    kernel=airline_periodic,
    C=periodic_parameters["C"],
    epsilon=periodic_parameters["epsilon"],
).fit(train_X, train_target)
print(f"Гладкость сезонного профиля: {periodic_parameters['length_scale']:g}")
periodic_result = evaluate_airline(periodic_forecaster, "Periodic")


Гладкость сезонного профиля: 2
Periodic: C=100, ε=0.01; полуширина полосы 0.9 тыс.


In [ ]:
plot_airline(periodic_result, "Periodic")
plt.tight_layout()
plt.show()


<Figure size 900x300 with 1 Axes>

**4. Тренд и сезонность.** В данных одновременно присутствуют рост среднего уровня и повторяющиеся сезонные колебания. Воспользуемся композиционным ядром
$$
\mathcal K=\mathcal K_{\mathrm{lin}}^{(t)}+\lambda\widetilde{\mathcal K}_{\mathrm{per}},\qquad \lambda>0.
$$
Ему соответствует составное отображение
$$
\psi_{\Sigma}(t)=\bigl(\psi_{\mathrm{lin}}(t),\sqrt\lambda\,\widetilde\psi_{\mathrm{per}}(t)\bigr),
\qquad f(t)=\langle w,\psi_{\Sigma}(t)\rangle+b.
$$
SVR обучает коэффициенты обоих блоков совместно. Поэтому заново выберем $C$, $\varepsilon$, $\ell_{\mathrm{per}}$ и $\lambda$ на валидации: лучшие параметры суммы не обязаны совпадать с параметрами отдельных ядер.

In [ ]:
def sum_kernel(X, Y, linear, period, length_scale, weight):
    seasonal = periodic_kernel(X, Y, period, length_scale)
    return linear(X, Y) + weight * seasonal


sum_candidates = []
sum_grid = {**periodic_grid, "weight": [0.1, 1, 10]}
for parameters in ParameterGrid(sum_grid):
    kernel = partial(
        sum_kernel,
        linear=fit_linear,
        period=season_period,
        length_scale=parameters["length_scale"],
        weight=parameters["weight"],
    )
    model = SVR(kernel=kernel, C=parameters["C"], epsilon=parameters["epsilon"]).fit(
        fit_X, fit_target
    )
    error = mean_absolute_error(validation_target, model.predict(validation_X))
    sum_candidates.append((error, parameters))
_, sum_parameters = min(sum_candidates, key=lambda item: item[0])

airline_sum = partial(
    sum_kernel,
    linear=airline_linear,
    period=season_period,
    length_scale=sum_parameters["length_scale"],
    weight=sum_parameters["weight"],
)
sum_forecaster = SVR(
    kernel=airline_sum, C=sum_parameters["C"], epsilon=sum_parameters["epsilon"]
).fit(train_X, train_target)
print(
    f"Гладкость сезонного профиля: {sum_parameters['length_scale']:g}; "
    f"вес сезонного ядра: {sum_parameters['weight']:g}"
)
sum_result = evaluate_airline(sum_forecaster, "Linear + Periodic")


Гладкость сезонного профиля: 0.5; вес сезонного ядра: 1
Linear + Periodic: C=10, ε=0.3; полуширина полосы 28.4 тыс.


In [ ]:
plot_airline(sum_result, "Linear + Periodic")
plt.tight_layout()
plt.show()


<Figure size 900x300 with 1 Axes>

**5. Линейный тренд и сезонность с растущей амплитудой.** Размах сезонных колебаний увеличивается со временем. Вычтем линейный тренд: $r_i=y_i-\hat a-\hat v t_i$. Для каждого года $j=1,\ldots,J$ объединим его 12 месяцев:
$$
I_j=\{i:j-1\leqslant t_i<j\},\qquad |I_j|=12.
$$
Оценим годовые амплитуды:
$$
A_j=\frac12\left(\max_{i\in I_j}r_i-\min_{i\in I_j}r_i\right),\qquad
\tau_j=\frac1{12}\sum_{i\in I_j}t_i,\qquad
\overline A=\frac1J\sum_{j=1}^J A_j.
$$
Оценим линейный рост амплитуды и определим соответствующий признак:
$$
(\hat a_A,\hat v_A)=\arg\min_{a,v}\sum_{j=1}^J(A_j-a-v\tau_j)^2,\qquad
\widehat A(t)=\hat a_A+\hat v_A t.
$$
$$
\psi_{\mathrm{amp}}(t)=\frac{\widehat A(t)}{\overline A}
=\frac{\hat a_A+\hat v_A t}{\overline A},\qquad
\frac1J\sum_{j=1}^J\psi_{\mathrm{amp}}(\tau_j)=1.
$$
$$
\mathcal K_{\mathrm{lin}}^{(a)}(t,s)=\psi_{\mathrm{amp}}(t)\psi_{\mathrm{amp}}(s).
$$
Умножим сезонные признаки на растущую амплитуду и добавим линейный тренд:
$$
\psi_{\mathrm{final}}(t)=\bigl(\psi_{\mathrm{lin}}(t),\sqrt\lambda\,\psi_{\mathrm{amp}}(t)\widetilde\psi_{\mathrm{per}}(t)\bigr),
$$
$$
\mathcal K=\mathcal K_{\mathrm{lin}}^{(t)}+\lambda\mathcal K_{\mathrm{lin}}^{(a)}\widetilde{\mathcal K}_{\mathrm{per}},
$$
$$
\begin{aligned}
f(t)&=\langle w,\psi_{\mathrm{final}}(t)\rangle+b\\
&=w_{\mathrm{lin}}\psi_{\mathrm{lin}}(t)
+\sqrt\lambda\,\psi_{\mathrm{amp}}(t)
\langle w_{\mathrm{per}},\widetilde\psi_{\mathrm{per}}(t)\rangle+b.
\end{aligned}
$$
Заново выберем $C$, $\varepsilon$, $\ell_{\mathrm{per}}$ и $\lambda$ на валидации; затем переоценим признаки и SVR по 1949–1958 годам.

In [ ]:
def amplitude_kernel(X, Y, slope, offset, mean):
    amplitude_X = (offset + slope * X) / mean
    amplitude_Y = (offset + slope * Y) / mean
    return amplitude_X @ amplitude_Y.T


fit_residual = fit_y - fit_trend_offset - fit_trend_slope * fit_X[:, 0]
fit_annual_amplitude = np.ptp(fit_residual.reshape(-1, 12), axis=1) / 2
fit_annual_time = fit_X.reshape(-1, 12).mean(axis=1)
fit_amplitude_slope, fit_amplitude_offset = np.polyfit(fit_annual_time, fit_annual_amplitude, 1)
fit_amplitude = partial(
    amplitude_kernel,
    slope=fit_amplitude_slope,
    offset=fit_amplitude_offset,
    mean=fit_annual_amplitude.mean(),
)
train_residual = train_y - trend_offset - trend_slope * train_X[:, 0]
annual_amplitude = np.ptp(train_residual.reshape(-1, 12), axis=1) / 2
annual_time = train_X.reshape(-1, 12).mean(axis=1)
amplitude_slope, amplitude_offset = np.polyfit(annual_time, annual_amplitude, 1)
airline_amplitude = partial(
    amplitude_kernel,
    slope=amplitude_slope,
    offset=amplitude_offset,
    mean=annual_amplitude.mean(),
)
print(f"Рост сезонной амплитуды: {amplitude_slope:.2f} тыс./год")

Рост сезонной амплитуды: 8.17 тыс./год


In [ ]:
def composition_kernel(X, Y, linear, amplitude, period, length_scale, weight):
    seasonal = amplitude(X, Y) * periodic_kernel(X, Y, period, length_scale)
    return linear(X, Y) + weight * seasonal


composition_candidates = []
for parameters in ParameterGrid(sum_grid):
    kernel = partial(
        composition_kernel,
        linear=fit_linear,
        amplitude=fit_amplitude,
        period=season_period,
        length_scale=parameters["length_scale"],
        weight=parameters["weight"],
    )
    model = SVR(kernel=kernel, C=parameters["C"], epsilon=parameters["epsilon"]).fit(
        fit_X, fit_target
    )
    error = mean_absolute_error(validation_target, model.predict(validation_X))
    composition_candidates.append((error, parameters))
_, composition_parameters = min(composition_candidates, key=lambda item: item[0])

airline_composition = partial(
    composition_kernel,
    linear=airline_linear,
    amplitude=airline_amplitude,
    period=season_period,
    length_scale=composition_parameters["length_scale"],
    weight=composition_parameters["weight"],
)
composition_forecaster = SVR(
    kernel=airline_composition,
    C=composition_parameters["C"],
    epsilon=composition_parameters["epsilon"],
).fit(train_X, train_target)
print(
    f"Гладкость сезонного профиля: {composition_parameters['length_scale']:g}; "
    f"вес сезонного ядра: {composition_parameters['weight']:g}"
)
composition_result = evaluate_airline(composition_forecaster, "Linear + Linear * Periodic")

Гладкость сезонного профиля: 0.25; вес сезонного ядра: 1
Linear + Linear * Periodic: C=100, ε=0.3; полуширина полосы 28.4 тыс.


In [ ]:
plot_airline(composition_result, "Linear + Linear * Periodic")
plt.tight_layout()
plt.show()


<Figure size 900x300 with 1 Axes>

**6. Оптимизация ширины трубы: $\nu$-SVR.** Добавим к функционалу $\varepsilon$-SVR плату за ширину трубы:
$$
J_\nu(w,b,\varepsilon)
=\frac12\|w\|^2
+C\left(\ell\nu\varepsilon+\sum_{i=1}^{\ell}
L_\varepsilon(\widetilde y_i,f(t_i))\right)
\longrightarrow\min_{w,b,\varepsilon\geqslant0},
\qquad 0<\nu\leqslant1.
$$
Здесь $\ell$ — число обучающих наблюдений; $C$ и $\nu$ фиксированы; $\varepsilon$ обучается вместе с $w,b$. Расширение трубы уменьшает штрафы за выходы, но увеличивает слагаемое $C\ell\nu\varepsilon$. Чем больше $\nu$, тем дороже ширина.

При фиксированном $f$ минимум по $\varepsilon$ достигается на $(1-\nu)$-квантиле абсолютных остатков $|\widetilde y_i-f(t_i)|$. По обучающим остаткам восстановим ширину для графика.

Зафиксируем итоговое ядро, $C=100$ и обучение по 1949–1958 годам. При $\nu=0.1$ сравним прежнюю модель с $\varepsilon=0.3$ и $\nu$-SVR по одному функционалу $J_{0.1}$. Функционал вычисляем на стандартизованной обучающей цели; его снижение сравниваем при одинаковом $\nu$.

In [ ]:
from sklearn.svm import NuSVR

fixed_C = composition_forecaster.C
nu = 0.1
nu_forecaster = NuSVR(kernel=airline_composition, C=fixed_C, nu=nu, tol=1e-7).fit(
    train_X, train_target
)

In [ ]:
def evaluate_tube(model, nu, epsilon):
    prediction = model.predict(train_X)
    residual = np.abs(train_target - prediction)
    support_X = train_X[model.support_]
    coefficients = model.dual_coef_[0]
    gram = airline_composition(support_X, support_X)
    norm_penalty = 0.5 * coefficients @ gram @ coefficients
    width_penalty = model.C * len(train_target) * nu * epsilon
    error_penalty = model.C * np.maximum(residual - epsilon, 0).sum()
    forecast = target_scaler.inverse_transform(model.predict(forecast_X)[:, None]).ravel()
    return {
        "Model": type(model).__name__, "nu": nu, "epsilon": epsilon,
        "Regularization": norm_penalty, "Width penalty": width_penalty,
        "Error penalty": error_penalty,
        "Objective": norm_penalty + width_penalty + error_penalty,
        "Forecast": forecast,
    }

In [ ]:
residual = np.abs(train_target - nu_forecaster.predict(train_X))
nu_epsilon = np.quantile(residual, 1 - nu)
tube_results = [
    evaluate_tube(composition_forecaster, nu, composition_forecaster.epsilon),
    evaluate_tube(nu_forecaster, nu, nu_epsilon),
]

tube_scores = pd.DataFrame(tube_results).drop(columns="Forecast")
tube_scores.round(3)

,Model,nu,epsilon,Regularization,Width penalty,Error penalty,Objective
0,SVR,0.1,0.300,1.011,360.000,3.538,364.55
1,NuSVR,0.1,0.233,1.120,279.008,48.471,328.60


In [ ]:
years = forecast_X[:, 0] + 1949
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharex=True, sharey=True, layout="constrained")
for ax, result in zip(axes, tube_results):
    forecast = result["Forecast"]
    epsilon = result["epsilon"] * target_scaler.scale_[0]
    ax.fill_between(
        years, forecast - epsilon, forecast + epsilon,
        color="royalblue", alpha=0.12, label=r"Полоса $\pm\varepsilon$",
    )
    for edge in [forecast - epsilon, forecast + epsilon]:
        ax.plot(years, edge, color="royalblue", linestyle="--", linewidth=0.8, alpha=0.6)
    ax.scatter(airline["time"][:120], train_y, color="gray", s=10, alpha=0.65, label="Обучение")
    ax.scatter(airline["time"][120:], test_y, color="darkorange", s=12, label="Тест")
    ax.plot(years, forecast, color="royalblue", label="Прогноз")
    ax.axvline(1959, color="black", linestyle="--", linewidth=1)
    ax.set(
        title=(
            f"{result['Model']}: ν={result['nu']:g}, ε={result['epsilon']:.3f}\n"
            + rf"$J_\nu^{{\mathrm{{train}}}}={result['Objective']:.1f}$"
        ),
        xlabel="Год", ylabel="Тысячи пассажиров", ylim=(50, 700),
    )
    ax.legend(fontsize=7, loc="upper left")
plt.show()

<Figure size 1000x320 with 2 Axes>

### Parkinson’s symptom severity

Эксперименты воспроизводят результаты статьи: https://pmc.ncbi.nlm.nih.gov/articles/PMC9543766

[Данные](https://archive.ics.uci.edu/dataset/189/parkinsons+telemonitoring) представляют собой характеристики голосовых записей пациентов с болезнью Паркинсона.

В выборке представлены 5875 записей 42 пациентов. Целевой показатель — суммарная оценка тяжести симптомов [Unified Parkinson’s Disease Rating Scale (UPDRS)](https://en.wikipedia.org/wiki/Unified_Parkinson%27s_disease_rating_scale), `total_UPDRS`.

In [ ]:
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile

parkinsons_url = "https://archive.ics.uci.edu/static/public/189/parkinsons+telemonitoring.zip"
with urlopen(parkinsons_url, timeout=15) as response:
    with ZipFile(BytesIO(response.read())) as archive:
        parkinsons = pd.read_csv(archive.open("parkinsons_updrs.data"))

X = parkinsons.drop(columns=["subject#", "motor_UPDRS", "total_UPDRS"])
y = parkinsons[["motor_UPDRS", "total_UPDRS"]]
X.columns


Index(['age', 'sex', 'test_time', 'Jitter(%)', 'Jitter(Abs)', 'Jitter:RAP',
       'Jitter:PPQ5', 'Jitter:DDP', 'Shimmer', 'Shimmer(dB)', 'Shimmer:APQ3',
       'Shimmer:APQ5', 'Shimmer:APQ11', 'Shimmer:DDA', 'NHR', 'HNR', 'RPDE',
       'DFA', 'PPE'],
      dtype='object')

In [ ]:
X

,age,sex,test_time,Jitter(%),Jitter(Abs),Jitter:RAP,Jitter:PPQ5,Jitter:DDP,Shimmer,Shimmer(dB),Shimmer:APQ3,Shimmer:APQ5,Shimmer:APQ11,Shimmer:DDA,NHR,HNR,RPDE,DFA,PPE
0,72,0,5.6431,0.00662,0.000034,0.00401,0.00317,0.01204,0.02565,0.230,0.01438,0.01309,0.01662,0.04314,0.014290,21.640,0.41888,0.54842,0.16006
1,72,0,12.6660,0.00300,0.000017,0.00132,0.00150,0.00395,0.02024,0.179,0.00994,0.01072,0.01689,0.02982,0.011112,27.183,0.43493,0.56477,0.10810
2,72,0,19.6810,0.00481,0.000025,0.00205,0.00208,0.00616,0.01675,0.181,0.00734,0.00844,0.01458,0.02202,0.020220,23.047,0.46222,0.54405,0.21014
3,72,0,25.6470,0.00528,0.000027,0.00191,0.00264,0.00573,0.02309,0.327,0.01106,0.01265,0.01963,0.03317,0.027837,24.445,0.48730,0.57794,0.33277
4,72,0,33.6420,0.00335,0.000020,0.00093,0.00130,0.00278,0.01703,0.176,0.00679,0.00929,0.01819,0.02036,0.011625,26.126,0.47188,0.56122,0.19361
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5870,61,0,142.7900,0.00406,0.000031,0.00167,0.00168,0.00500,0.01896,0.160,0.00973,0.01133,0.01549,0.02920,0.025137,22.369,0.64215,0.55314,0.21367
5871,61,0,149.8400,0.00297,0.000025,0.00119,0.00147,0.00358,0.02315,0.215,0.01052,0.01277,0.01904,0.03157,0.011927,22.886,0.52598,0.56518,0.12621
5872,61,0,156.8200,0.00349,0.000025,0.00152,0.00187,0.00456,0.02499,0.244,0.01371,0.01456,0.01877,0.04112,0.017701,25.065,0.47792,0.57888,0.14157
5873,61,0,163.7300,0.00281,0.000020,0.00128,0.00151,0.00383,0.01484,0.131,0.00693,0.00870,0.01307,0.02078,0.007984,24.422,0.56865,0.56327,0.14204


In [ ]:
y

,motor_UPDRS,total_UPDRS
0,28.199,34.398
1,28.447,34.894
2,28.695,35.389
3,28.905,35.810
4,29.187,36.375
...,...,...
5870,22.485,33.485
5871,21.988,32.988
5872,21.495,32.495
5873,21.007,32.007


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

y = y["total_UPDRS"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

Сравним SVR с RBF-ядром, суммой и произведением RBF и полиномиального ядра:
$$
\mathcal K_{\mathrm{rbf}}(x,s)=e^{-\gamma\|x-s\|^2},\qquad
\mathcal K_{\mathrm{poly}}(x,s)=(1+x^\top s)^2.
$$
Отложим 20% записей в test. Для всех трёх моделей зададим $C=1$, $\varepsilon=0.1$ балла UPDRS и $\gamma=1/d$, где $d=19$ — число стандартизованных признаков. Сравним MAE и RMSE на тесте.

#### SVR with RBF

In [ ]:
from sklearn.metrics.pairwise import polynomial_kernel, rbf_kernel

parkinson_gamma = 1 / X_train_scaled.shape[1]
parkinson_rbf = SVR(kernel="rbf", gamma=parkinson_gamma, C=1, epsilon=0.1)
parkinson_rbf.fit(X_train_scaled, y_train)
parkinson_predictions = {"RBF": parkinson_rbf.predict(X_test_scaled)}
mean_absolute_error(y_test, parkinson_predictions["RBF"])

6.615092620331981

#### SVR with (RBF + Poly)

$$\mathcal K=\tfrac12\bigl(\mathcal K_{\mathrm{rbf}}+\mathcal K_{\mathrm{poly}}\bigr).$$

In [ ]:
def mixed_kernel_sum(X, Y, gamma):
    radial = rbf_kernel(X, Y, gamma=gamma)
    polynomial = polynomial_kernel(X, Y, degree=2, gamma=1, coef0=1)
    return 0.5 * (radial + polynomial)


parkinson_sum = SVR(
    kernel=partial(mixed_kernel_sum, gamma=parkinson_gamma), C=1, epsilon=0.1
)
parkinson_sum.fit(X_train_scaled, y_train)
parkinson_predictions["RBF + Poly"] = parkinson_sum.predict(X_test_scaled)
mean_absolute_error(y_test, parkinson_predictions["RBF + Poly"])

6.815310437914994

#### SVR with (RBF * Poly)

$$\mathcal K=\mathcal K_{\mathrm{rbf}}\mathcal K_{\mathrm{poly}}.$$

In [ ]:
def mixed_kernel_product(X, Y, gamma):
    radial = rbf_kernel(X, Y, gamma=gamma)
    polynomial = polynomial_kernel(X, Y, degree=2, gamma=1, coef0=1)
    return radial * polynomial


parkinson_product = SVR(
    kernel=partial(mixed_kernel_product, gamma=parkinson_gamma), C=1, epsilon=0.1
)
parkinson_product.fit(X_train_scaled, y_train)
parkinson_predictions["RBF * Poly"] = parkinson_product.predict(X_test_scaled)
parkinson_scores = pd.DataFrame(
    {
        name: {
            "Test MAE": mean_absolute_error(y_test, prediction),
            "Test RMSE": np.sqrt(mean_squared_error(y_test, prediction)),
        }
        for name, prediction in parkinson_predictions.items()
    }
).T
parkinson_scores.round(3)

,Test MAE,Test RMSE
RBF,6.615,8.871
RBF + Poly,6.815,9.661
RBF * Poly,4.563,6.910


In [ ]:
limits = (y_train.min(), y_train.max())
fig, axes = plt.subplots(1, 3, figsize=(10, 3.3), layout="constrained")
for ax, (name, prediction) in zip(axes, parkinson_predictions.items()):
    error = parkinson_scores.loc[name, "Test MAE"]
    ax.scatter(y_test, prediction, s=8, alpha=0.3)
    ax.plot(limits, limits, color="black", linestyle="--", linewidth=1)
    ax.set(
        title=f"{name}: MAE = {error:.2f}",
        xlabel="UPDRS", ylabel="Прогноз UPDRS",
        xlim=limits, ylim=limits, aspect="equal",
    )
plt.show()

<Figure size 1000x330 with 3 Axes>

Произведение ядер уменьшило MAE с $6.62$ до $4.56$. В отличие от AirPassengers, здесь по исходным признакам нельзя непосредственно выделить тренд и сезонность и на их основе выбрать ядро; композиции сравниваем по ошибке прогноза.
